# 1. 데이터 로드 및 전처리

## 1.1 필요한 라이브러리 불러오기

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
# 데이터가 저장된 기본 경로
base_path = 'datasets/test_datasets'


## 1.2 Parquet 파일 불러오기 및 전처리

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm

def load_parquet_files(file_paths):
    """Load multiple parquet files into a single pandas DataFrame."""
    return pd.concat([pd.read_parquet(file) for file in tqdm(file_paths)])

def load_parquet_data(base_path):
    """Load all required parquet datasets from a specified base path."""
    acc_files = [
        'ch2024_test__m_acc_part_5.parquet.gzip',
        'ch2024_test__m_acc_part_6.parquet.gzip',
        'ch2024_test__m_acc_part_7.parquet.gzip',
        'ch2024_test__m_acc_part_8.parquet.gzip'
    ]
    
    acc_file_paths = [f'{base_path}/{file}' for file in acc_files]
    acc_data = load_parquet_files(acc_file_paths)

    data_files = {
        'activity_data': 'ch2024_test_m_activity.parquet.gzip',
        'hr_data': 'ch2024_test_w_heart_rate.parquet.gzip',
        'step_count': 'ch2024_test_w_pedo.parquet.gzip',
        'light': 'ch2024_test_m_light.parquet.gzip',
        'gps': 'ch2024_test_m_gps.parquet.gzip',
    }

    data = {name: pd.read_parquet(f'{base_path}/{file}') for name, file in data_files.items()}
    data['acc_data'] = acc_data  # Include acc_data in the main dictionary
    return data

def preprocess_data(data):
    """Process all datasets: reset index, add date columns, and handle specific value replacements."""
    processed_data = {}
    for name, df in data.items():
        df = df.reset_index(drop=True)
        
        if 'timestamp' in df.columns:
            df['date'] = df['timestamp'].dt.date
        
        if name == 'hr_data':
            df['heart_rate'].replace(0, np.nan, inplace=True)
        
        processed_data[name] = df
    
    return processed_data

def load_and_preprocess_data(base_path):
    """Load and preprocess all data from the base path."""
    data = load_parquet_data(base_path)
    return preprocess_data(data)

# Set your base path to the location where your files are stored.

# Load and preprocess data
data = load_and_preprocess_data(base_path)
print('Data loading and preprocessing completed.')

# Access specific datasets like this:
acc_data = data['acc_data']
activity_data = data['activity_data']
hr_data = data['hr_data']
step_count = data['step_count']
light = data['light']
gps = data['gps']


100% 4/4 [01:27<00:00, 21.96s/it]


Data loading and preprocessing completed.


In [3]:
acc_data

,subject_id,timestamp,x,y,z,date
0,5,2023-11-05 00:00:00.022,-0.088585,0.105345,9.749489,2023-11-05
1,5,2023-11-05 00:00:00.039,-0.067038,0.112527,9.742306,2023-11-05
2,5,2023-11-05 00:00:00.136,-0.071826,0.100556,9.742306,2023-11-05
3,5,2023-11-05 00:00:00.202,-0.059855,0.081403,9.672874,2023-11-05
4,5,2023-11-05 00:00:00.218,-0.083797,0.110133,9.720758,2023-11-05
...,...,...,...,...,...,...
489391448,8,2023-11-09 23:59:59.915,0.055068,0.260976,9.457385,2023-11-09
489391449,8,2023-11-09 23:59:59.932,0.050280,0.260976,9.715967,2023-11-09
489391450,8,2023-11-09 23:59:59.953,0.062251,0.256187,9.586677,2023-11-09
489391451,8,2023-11-09 23:59:59.975,0.062251,0.265765,9.608225,2023-11-09


In [4]:
activity_data

,subject_id,timestamp,m_activity,date
0,5,2023-11-05 00:00:39.044,3,2023-11-05
1,5,2023-11-05 00:01:39.044,3,2023-11-05
2,5,2023-11-05 00:02:39.044,3,2023-11-05
3,5,2023-11-05 00:03:39.043,3,2023-11-05
4,5,2023-11-05 00:04:39.044,3,2023-11-05
...,...,...,...,...
165063,8,2023-11-09 23:55:27.604,3,2023-11-09
165064,8,2023-11-09 23:56:27.604,3,2023-11-09
165065,8,2023-11-09 23:57:27.604,3,2023-11-09
165066,8,2023-11-09 23:58:27.604,3,2023-11-09


In [5]:
hr_data

,subject_id,timestamp,heart_rate,date
0,5,2023-11-05 00:00:38.081,90.0,2023-11-05
1,5,2023-11-05 00:01:38.248,65.0,2023-11-05
2,5,2023-11-05 00:02:38.340,86.0,2023-11-05
3,5,2023-11-05 00:03:38.450,89.0,2023-11-05
4,5,2023-11-05 00:07:20.468,80.0,2023-11-05
...,...,...,...,...
132777,8,2023-11-09 23:55:00.517,76.0,2023-11-09
132778,8,2023-11-09 23:56:00.524,85.0,2023-11-09
132779,8,2023-11-09 23:57:01.521,81.0,2023-11-09
132780,8,2023-11-09 23:58:02.520,78.0,2023-11-09


In [6]:
step_count

,subject_id,timestamp,burned_calories,distance,running_steps,speed,steps,step_frequency,walking_steps,date
0,5,2023-11-05 00:00:00,0.0,7.73,0,0.128833,10,0.166667,0,2023-11-05
1,5,2023-11-05 00:01:00,0.0,0.00,0,0.000000,0,0.000000,0,2023-11-05
2,5,2023-11-05 00:02:00,0.0,0.00,0,0.000000,0,0.000000,0,2023-11-05
3,5,2023-11-05 00:03:00,0.0,0.00,0,0.000000,0,0.000000,0,2023-11-05
4,5,2023-11-05 00:04:00,0.0,0.00,0,0.000000,0,0.000000,0,2023-11-05
...,...,...,...,...,...,...,...,...,...,...
119806,8,2023-11-09 23:55:00,0.0,0.00,0,0.000000,0,0.000000,0,2023-11-09
119807,8,2023-11-09 23:56:00,0.0,0.00,0,0.000000,0,0.000000,0,2023-11-09
119808,8,2023-11-09 23:57:00,0.0,0.00,0,0.000000,0,0.000000,0,2023-11-09
119809,8,2023-11-09 23:58:00,0.0,0.00,0,0.000000,0,0.000000,0,2023-11-09


In [7]:
light

,subject_id,timestamp,m_light,date
0,5,2023-11-05 00:09:00,208.0,2023-11-05
1,5,2023-11-05 00:19:00,212.0,2023-11-05
2,5,2023-11-05 00:29:00,211.0,2023-11-05
3,5,2023-11-05 00:39:00,194.0,2023-11-05
4,5,2023-11-05 00:49:00,214.0,2023-11-05
...,...,...,...,...
16274,8,2023-11-09 23:22:00,0.0,2023-11-09
16275,8,2023-11-09 23:29:00,18.0,2023-11-09
16276,8,2023-11-09 23:39:00,107.0,2023-11-09
16277,8,2023-11-09 23:49:00,107.0,2023-11-09


In [8]:
gps

,subject_id,timestamp,altitude,latitude,longitude,speed,date
0,5,2023-11-05 02:25:01,95.734328,0.038370,0.028696,0.126473,2023-11-05
1,5,2023-11-05 02:25:06,95.734328,0.038373,0.028697,0.032710,2023-11-05
2,5,2023-11-05 02:25:09,95.734328,0.038373,0.028704,0.237968,2023-11-05
3,5,2023-11-05 02:25:12,95.734328,0.038367,0.028711,0.146265,2023-11-05
4,5,2023-11-05 02:25:15,95.734328,0.038362,0.028713,0.051259,2023-11-05
...,...,...,...,...,...,...,...
746712,8,2023-11-09 23:33:41,138.599991,1.594249,0.929489,0.051794,2023-11-09
746713,8,2023-11-09 23:33:46,138.500000,1.594255,0.929493,0.144418,2023-11-09
746714,8,2023-11-09 23:33:51,138.500000,1.594263,0.929493,0.161056,2023-11-09
746715,8,2023-11-09 23:33:57,138.500000,1.594303,0.929534,1.045928,2023-11-09


# parquet 저장 (user, date 별)

In [9]:
def save_grouped_data(df, file_prefix):
    for (subject_id, date), group in df.groupby(['subject_id', 'date']):
        directory = f'datasets/raw_datasets_all_sensor/test_datasets/{file_prefix}'
        if not os.path.exists(directory):
            os.makedirs(directory)
        file_path = f'{directory}/user{subject_id}_{date}.parquet'
        group.to_parquet(file_path)

acc_data = data['acc_data']
activity_data = data['activity_data']
hr_data = data['hr_data']
step_count = data['step_count']
light = data['light']
gps = data['gps']
activity_data['m_activity'] = activity_data['m_activity'].fillna(4).astype(int)

save_grouped_data(data['acc_data'], 'acc')
save_grouped_data(data['activity_data'], 'activity')
save_grouped_data(data['hr_data'], 'hr')
save_grouped_data(data['step_count'], 'step')
save_grouped_data(data['light'], 'light')
save_grouped_data(data['gps'], 'gps')

# 2. 데이터 병합

In [10]:
import pandas as pd
from datetime import datetime, timedelta

def make_timestamps_unique(df, timestamp_col='timestamp'):
    if not df.empty:
        df = df.sort_values(by=[timestamp_col])
        df[timestamp_col] = df[timestamp_col] + pd.to_timedelta(df.groupby(timestamp_col).cumcount(), unit='ns')
    return df

def merge_data_for_group(user, date):
    data_sources = {}
    sensor_files = ['acc', 'activity', 'hr', 'step', 'light', 'gps']
    
    for sensor in sensor_files:
        file_path = f'datasets/raw_datasets_all_sensor/test_datasets/{sensor}/user{user}_{date}.parquet'
        try:
            df = pd.read_parquet(file_path)
            if sensor == 'gps':
                df = df.drop('speed', axis=1)
                
            if sensor == 'activity':
                df['m_activity'] = df['m_activity'].astype(int)
            # 삭제할 컬럼이 없는 경우를 처리하기 위해 errors='ignore' 추가
            df = df.drop(columns=['subject_id', 'date'], errors='ignore')
            data_sources[sensor] = df
        except FileNotFoundError:
            columns_map = {
                'acc': ['timestamp', 'x', 'y', 'z'],
                'activity': ['timestamp', 'm_activity'],
                'hr': ['timestamp', 'heart_rate'],
                'step': ['timestamp', 'steps'],
                'light': ['timestamp', 'm_light'],
                'gps': ['timestamp', 'altitude', 'latitude', 'longitude']
            }
            data_sources[sensor] = pd.DataFrame(columns=columns_map[sensor])
        
        if not data_sources[sensor].empty:
            data_sources[sensor]['timestamp'] = pd.to_datetime(data_sources[sensor]['timestamp'])
            data_sources[sensor] = make_timestamps_unique(data_sources[sensor])
            data_sources[sensor] = data_sources[sensor].set_index('timestamp').resample('S').nearest()
        else:
            data_sources[sensor] = pd.DataFrame(index=pd.date_range(start=datetime.strptime(date, '%Y-%m-%d'), periods=86400, freq='S'))

    merged_data = pd.DataFrame(index=pd.date_range(start=datetime.strptime(date, '%Y-%m-%d'), periods=86400, freq='S'))
    for key, df in data_sources.items():
        merged_data = merged_data.join(df, how='left', rsuffix='_duplicate')  # 중복 컬럼명에 접미사 '_duplicate' 추가

    # 필요한 경우, 중복 컬럼 삭제
    duplicate_cols = [col for col in merged_data.columns if '_duplicate' in col]
    merged_data = merged_data.drop(columns=duplicate_cols)
    y_ranges = {}
    for column in merged_data.columns:
        if not merged_data[column].dropna().empty:  # 신뢰구간 계산 전 NaN 제거
            y_ranges[column] = (
                merged_data[column].quantile(0.0001, interpolation='linear'),
                merged_data[column].quantile(0.9999, interpolation='linear')
            )

    # 데이터 보간
    merged_data = merged_data.interpolate(method='linear')

    return merged_data, y_ranges



# 3 시각화

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

def plot_time_series(data, user, date, y_ranges):
    # x축을 00:00:00부터 23:59:59까지 고정
    total_seconds = 86400
    time_range = pd.date_range(start=datetime.strptime(date, '%Y-%m-%d'), periods=total_seconds, freq='S')

    # 데이터를 시간 단위로 정렬
    data = data.reindex(time_range)

    # 시계열 이미지 생성
    fig, axes = plt.subplots(11, 1, figsize=(10, 10), sharex=True, facecolor='black')
    fig.patch.set_facecolor('black')

    for ax in axes:
        ax.set_facecolor('black')
        ax.spines['top'].set_visible(False)   # Hide the top spine
        ax.spines['right'].set_visible(False) # Hide the right spine
        ax.spines['left'].set_visible(False)  # Hide the left spine
        ax.spines['bottom'].set_visible(False)# Hide the bottom spine

    # 설정한 시간 범위에 맞게 x축 설정
    for ax in axes:
        ax.set_xlim([time_range[0], time_range[-1]])

    # 컬럼 인덱스 딕셔너리 생성
    column_indices = {
        'x': 0, 'y': 1, 'z': 2, 'heart_rate': 3, 'm_activity': 4,
        'speed': 5, 'steps': 6,'m_light': 7, 'altitude': 8, 'latitude': 9, 'longitude': 10
    }

    # 각 컬럼에 대한 그래프 플로팅 및 y축 범위 설정
    for column, idx in column_indices.items():
        if column in data.columns:
            axes[idx].plot(data.index, data[column], color='white')
#             axes[idx].set_ylabel(column, color='white')  # 레이블 설정
            # y축 범위 설정
            if column in y_ranges:
                if column == 'm_activity':
                    axes[idx].set_ylim((0, 2))
                else:
                    axes[idx].set_ylim(y_ranges[column])

    plt.tight_layout()  # Make the layout tight
    plt.savefig(f'datasets/image_datasets_all_sensor/user{user}_{date}_test_all_sensor.png')
#     plt.show()


# 3. 데이터셋 생성 (Valid)

## 3.1 Valid data 

In [12]:
test_df = pd.read_csv('datasets/answer_sample.csv')
test_df

,subject_id,date,Q1,Q2,Q3,S1,S2,S3,S4
0,5,2023-11-05,-1,-1,-1,-1,-1,-1,-1
1,5,2023-11-06,-1,-1,-1,-1,-1,-1,-1
2,5,2023-11-07,-1,-1,-1,-1,-1,-1,-1
3,5,2023-11-08,-1,-1,-1,-1,-1,-1,-1
4,5,2023-11-09,-1,-1,-1,-1,-1,-1,-1
...,...,...,...,...,...,...,...,...,...
110,8,2023-11-05,-1,-1,-1,-1,-1,-1,-1
111,8,2023-11-06,-1,-1,-1,-1,-1,-1,-1
112,8,2023-11-07,-1,-1,-1,-1,-1,-1,-1
113,8,2023-11-08,-1,-1,-1,-1,-1,-1,-1


In [ ]:
from tqdm.auto import tqdm

bar = tqdm(range(test_df.shape[0]))

for idx in bar:
    user, date, *rest = test_df.iloc[idx].values
    bar.set_description(f'user: {user}, date: {date}')
    merged_data, y_ranges = merge_data_for_group(user, date)
    plot_time_series(merged_data, user, date, y_ranges)